# Teaching an LLM to Solve Math Problems with RL on Amazon SageMaker Training Sessions

## Part 2: Deploy

This is part 2 of the walkthrough (see `sm-training_sessions_grpo_gsm8k.ipynb` for part 1). Here we take the checkpoint
`sm-training_sessions_grpo_gsm8k.ipynb` saved and deploy it to a **SageMaker
real-time endpoint**, then send it a math question.

### Deployment architecture

SageMaker real-time endpoints use an **Inference Component** pattern that decouples the
endpoint's infrastructure from the model hosted on it:

1. **Endpoint Configuration** — defines the instance type
2. **Endpoint** — provisions the compute infrastructure
3. **Model** — registers the fine-tuned weights with a serving container (DJL LMI + vLLM)
4. **Inference Component** — attaches the model to the endpoint with specific resource
   requirements

> **Note:** deploy a **merged** checkpoint — one saved with
> `save_weights_for_sampler(merge_weights=True)` in `sm-training_sessions_grpo_gsm8k.ipynb`, a self-contained set of
> weights. A plain `save_weights_for_sampler()` registers only the LoRA adapter, and a
> `save_state` checkpoint holds sharded training state for resuming a run — neither is
> deployable as a single model here.
>
> **Before running:** Section 2 prints the checkpoint's model data location, the container
> image, and the instance type. The instance type is sized for GPT-OSS 20B; change it if you
> trained a different base model.

## 1. Load config

Same auto-discovery as `sm-training_sessions_grpo_gsm8k.ipynb` -- no config file. `MODEL_PACKAGE_GROUP_NAME` is the
same fixed constant `sm-training_sessions_grpo_gsm8k.ipynb` uses, so the ARN it builds here matches without needing to
be passed between notebooks.

In [ ]:
import json

import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="SageMakerTrainingSessionRole")["Role"]["Arn"]

account_id = sess.account_id()
REGION = sess.boto_region_name

# Must match the constant of the same name in sm-training_sessions_grpo_gsm8k.ipynb -- the ARN is deterministic
# from account_id + REGION + this name, so no config file is needed to hand it off.
MODEL_PACKAGE_GROUP_NAME = "gsm8k-grpo-training-sessions"
MODEL_PACKAGE_GROUP_ARN = (
    f"arn:aws:sagemaker:{REGION}:{account_id}:model-package-group/{MODEL_PACKAGE_GROUP_NAME}"
)

sm = boto3.client("sagemaker", region_name=REGION)
sm_runtime = boto3.client("sagemaker-runtime", region_name=REGION)

print(f"account id: {account_id}")
print(f"role arn:   {role}")
print(f"region:     {REGION}")
print(f"model_package_group_arn: {MODEL_PACKAGE_GROUP_ARN}")

## 2. Find the checkpoint to deploy

List the checkpoints in the Model Package Group `sm-training_sessions_grpo_gsm8k.ipynb` created, and find the most
recent deployable one. `list_model_packages(kind=ModelPackageKind.MODEL)` filters out the
`save_state` checkpoints for us (it describes candidates server-side to resolve their kind), and
we then keep only the packages that were *merged*. Each package's S3 weights location is read
from its model package description.

In [ ]:
from sagemaker.train.training_session import ModelPackageKind, ServiceClient

# Platform-level parameters live on the ServiceClient: role_arn is required, and the model
# package group set here is the default for the lookups below.
svc = ServiceClient(
    region=REGION,
    role_arn=role,
    model_package_group_arn=MODEL_PACKAGE_GROUP_ARN,
)
s3 = boto3.client("s3", region_name=REGION)

# Set this manually to deploy a specific checkpoint instead (skips the auto-selection below).
deployable_checkpoint_arn = None


def _model_data_root(container: dict):
    # The weights S3 root lives under either the newer ModelDataSource.S3DataSource.S3Uri
    # or the legacy ModelDataUrl -- handle both.
    if "ModelDataSource" in container:
        return container["ModelDataSource"]["S3DataSource"]["S3Uri"]
    return container.get("ModelDataUrl")


def _has_merged_weights(root: str) -> bool:
    # save_weights_for_sampler(merge_weights=True) writes the deployable, self-contained model
    # under <root>/checkpoints/hf_merged/. A non-merged save has no such prefix.
    bucket, _, key = root[len("s3://") :].partition("/")
    prefix = key.rstrip("/") + "/checkpoints/hf_merged/"
    resp = s3.list_objects_v2(Bucket=bucket, Prefix=prefix, MaxKeys=1)
    return resp.get("KeyCount", 0) > 0


# kind=MODEL keeps only save_weights_for_sampler packages (save_state ones are sharded training
# state, not deployable), newest first. Whether a MODEL was *merged* isn't in the API, so we
# still probe S3 for checkpoints/hf_merged/ -- the newest package is often a per-step,
# non-merged save from the training loop.
checkpoints = svc.list_model_packages(kind=ModelPackageKind.MODEL, max_results=20)

print(f"{'Merged':<8} {'Model package ARN'}")
print("-" * 100)
auto_arn = None
for ckpt in checkpoints:
    pkg = sm.describe_model_package(ModelPackageName=ckpt.model_package_arn)
    container = pkg["InferenceSpecification"]["Containers"][0]
    root = _model_data_root(container)
    merged = "yes" if root and _has_merged_weights(root) else "no"
    print(f"{merged:<8} {ckpt.model_package_arn}")
    if auto_arn is None and merged == "yes":
        auto_arn = ckpt.model_package_arn  # newest deployable (merged) model

if deployable_checkpoint_arn is None:
    if auto_arn is None:
        raise RuntimeError(
            "No merged, deployable checkpoint found. In sm-training_sessions_grpo_gsm8k.ipynb, make sure "
            "training_client.save_weights_for_sampler(merge_weights=True) ran -- that is what "
            "writes checkpoints/hf_merged/. Per-step (non-merged) saves and save_state "
            "checkpoints are not deployable."
        )
    deployable_checkpoint_arn = auto_arn

print("\nDeploying checkpoint:", deployable_checkpoint_arn)

In [ ]:
# Pull the real weights location from the model package itself. describe_model_package
# can return the S3 URI under either ModelDataUrl (legacy, single field) or the newer
# ModelDataSource.S3DataSource.S3Uri -- handle both and print what we found so you can
# eyeball it before it's used to create billable infrastructure below.
pkg = sm.describe_model_package(ModelPackageName=deployable_checkpoint_arn)
container = pkg["InferenceSpecification"]["Containers"][0]

if "ModelDataSource" in container:
    model_root = container["ModelDataSource"]["S3DataSource"]["S3Uri"]
else:
    model_root = container["ModelDataUrl"]

# save_weights_for_sampler(merge_weights=True) writes the merged, self-contained model under the
# model package's checkpoints/hf_merged/ prefix -- deploy that, not the package root.
weights_s3_uri = model_root.rstrip("/") + "/checkpoints/hf_merged/"

print("Weights S3 URI:", weights_s3_uri)
print("\nFull container definition (for reference):")
print(json.dumps(container, indent=2, default=str))

## 3. Create the endpoint

An **Endpoint Configuration** specifies the instance type; the **Endpoint** provisions it.
`ml.g5.12xlarge` (4x A10G) below is sized for a ~20B model like GPT-OSS-20B -- turn it down for
a smaller base model, or up for a larger one.

> **⏱ Expected duration:** the endpoint takes a few minutes to reach `InService`.

In [ ]:
import time

MODEL_NAME = f"gsm8k-training-sessions-{int(time.time())}"
ENDPOINT_CONFIG_NAME = f"{MODEL_NAME}-config"
ENDPOINT_NAME = f"{MODEL_NAME}-endpoint"
IC_NAME = f"{MODEL_NAME}-ic"

INSTANCE_TYPE = "ml.g5.12xlarge"  # adjust to your base model's size
NUM_ACCELERATORS = 4  # GPUs on the instance type above

print(f"Creating EndpointConfig: {ENDPOINT_CONFIG_NAME}")
sm.create_endpoint_config(
    EndpointConfigName=ENDPOINT_CONFIG_NAME,
    ExecutionRoleArn=role,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "InstanceType": INSTANCE_TYPE,
            "InitialInstanceCount": 1,
            "ModelDataDownloadTimeoutInSeconds": 900,
            "RoutingConfig": {"RoutingStrategy": "LEAST_OUTSTANDING_REQUESTS"},
        }
    ],
)

In [ ]:
print(f"Creating Endpoint: {ENDPOINT_NAME}")
sm.create_endpoint(EndpointName=ENDPOINT_NAME, EndpointConfigName=ENDPOINT_CONFIG_NAME)

waiter = sm.get_waiter("endpoint_in_service")
waiter.wait(EndpointName=ENDPOINT_NAME)
print(f"Endpoint {ENDPOINT_NAME} is InService")

## 4. Create the model and inference component

We register the checkpoint using the **DJL LMI** (Large Model Inference) container with
**vLLM** as the serving backend. `image_uris.retrieve` resolves the container image for your Region from the
LMI version. See the
[LMI container release notes](https://github.com/deepjavalibrary/djl-serving/releases)
for newer versions.

In [ ]:
from sagemaker.core import image_uris

LMI_VERSION = "0.36.0"
inference_image = image_uris.retrieve("djl-lmi", region=REGION, version=LMI_VERSION)

lmi_env = {
    "SERVING_FAIL_FAST": "true",
    "OPTION_ASYNC_MODE": "true",
    "OPTION_ROLLING_BATCH": "disable",
    "OPTION_MAX_MODEL_LEN": "16384",
    "OPTION_TENSOR_PARALLEL_DEGREE": "max",
    "OPTION_ENTRYPOINT": "djl_python.lmi_vllm.vllm_async_service",
    "OPTION_TRUST_REMOTE_CODE": "true",
}

print("Image:", inference_image)
print("Weights:", weights_s3_uri)

In [ ]:
print(f"Creating Model: {MODEL_NAME}")
sm.create_model(
    ModelName=MODEL_NAME,
    ExecutionRoleArn=role,
    PrimaryContainer={
        "Image": inference_image,
        "ModelDataSource": {
            "S3DataSource": {
                "S3Uri": weights_s3_uri,
                "S3DataType": "S3Prefix",
                "CompressionType": "None",
            }
        },
        "Environment": lmi_env,
    },
)

In [ ]:
import time

print(f"Creating InferenceComponent: {IC_NAME}")
sm.create_inference_component(
    InferenceComponentName=IC_NAME,
    EndpointName=ENDPOINT_NAME,
    VariantName="AllTraffic",
    Specification={
        "ModelName": MODEL_NAME,
        "ComputeResourceRequirements": {
            "MinMemoryRequiredInMb": 10240,
            "NumberOfAcceleratorDevicesRequired": NUM_ACCELERATORS,
        },
    },
    RuntimeConfig={"CopyCount": 1},
)

while True:
    status = sm.describe_inference_component(InferenceComponentName=IC_NAME)[
        "InferenceComponentStatus"
    ]
    if status == "InService":
        break
    if status == "Failed":
        reason = sm.describe_inference_component(InferenceComponentName=IC_NAME).get(
            "FailureReason"
        )
        raise RuntimeError(f"InferenceComponent {IC_NAME} entered Failed: {reason}")
    print(f"  status={status} ... waiting")
    time.sleep(15)
print(f"InferenceComponent {IC_NAME} is InService")

## 5. Test the endpoint

Send it a GSM8K-style math question and see if it gets the right answer.

In [ ]:
question = (
    "Natalia sold clips to 48 of her friends in April, and then she sold half as many "
    "clips in May. How many clips did Natalia sell altogether in April and May?"
)
prompt = (
    f"Question: {question}\n"
    "Solve this step by step, then give the final numeric answer on its own line "
    "after '#### '.\nAnswer:"
)

payload = {
    "inputs": prompt,
    "parameters": {"max_new_tokens": 256, "temperature": 0.1, "top_p": 0.9},
}

response = sm_runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    InferenceComponentName=IC_NAME,
    ContentType="application/json",
    Body=json.dumps(payload),
)
result = json.loads(response["Body"].read())
print(result)

## 6. Clean up

Real-time endpoints are billed for as long as they're running. Delete in order: inference
component, model, endpoint, endpoint config.

In [ ]:
import time
from botocore.exceptions import ClientError

sm.delete_inference_component(InferenceComponentName=IC_NAME)
# No inference_component_deleted waiter in botocore -- poll until Describe says it's gone.
while True:
    try:
        sm.describe_inference_component(InferenceComponentName=IC_NAME)
        time.sleep(10)
    except ClientError as e:
        if e.response["Error"]["Code"] in ("ValidationException", "ResourceNotFound"):
            break
        raise
print(f"Deleted inference component: {IC_NAME}")

sm.delete_model(ModelName=MODEL_NAME)
print(f"Deleted model: {MODEL_NAME}")

sm.delete_endpoint(EndpointName=ENDPOINT_NAME)
sm.get_waiter("endpoint_deleted").wait(EndpointName=ENDPOINT_NAME)
print(f"Deleted endpoint: {ENDPOINT_NAME}")

sm.delete_endpoint_config(EndpointConfigName=ENDPOINT_CONFIG_NAME)
print(f"Deleted endpoint config: {ENDPOINT_CONFIG_NAME}")